# 02 : nettoyage

Application des décisions prises dans `01_exploration.ipynb`. Une cellule de journal par décision : quoi, pourquoi, combien de lignes touchées. Les fichiers bruts ne sont jamais modifiés : le résultat est enregistré dans `data/processed/`.

In [1]:
import sys

import pandas as pd

sys.path.append("..")
from src.nettoyage import harmoniser_organisations, normaliser_liste, parle_d_ia, retirer_parametres_de_suivi

articles = pd.read_csv(
    "../data/raw/hacker_news_top_stories_2020_2026.csv",
    parse_dates=["created_at"],
)
modeles = pd.read_csv(
    "../data/raw/ai_models_panel.csv",
    parse_dates=["publication_date", "last_modified"],
)

print(f"Articles : {articles.shape}, modèles : {modeles.shape}")

Articles : (100286, 11), modèles : (3626, 28)


## 1. Hacker News

### Colonnes inutiles

In [2]:
articles = articles.drop(columns=["story_type", "created_at_i", "year"])
articles.columns.tolist()

['id',
 'title',
 'url',
 'domain',
 'author',
 'score',
 'num_comments',
 'created_at']

| Quoi | Pourquoi | Lignes |
|---|---|---|
| `story_type`, `created_at_i` et `year` supprimées | `story_type` ne prend qu'une valeur (« story »), `created_at_i` et `year` répètent `created_at` à 100 % | 100 286 |

### Articles sans lien externe

In [3]:
sans_url = articles["url"].isna()
part_ask_hn = articles.loc[sans_url, "title"].str.match(r"(?:Ask|Tell|Launch) HN").mean()

print(f"Articles sans URL : {sans_url.sum()}, dont {part_ask_hn:.0%} de Ask, Tell ou Launch HN")
articles["domain"] = articles["domain"].fillna("news.ycombinator.com")

Articles sans URL : 3466, dont 86% de Ask, Tell ou Launch HN


| Quoi | Pourquoi | Lignes |
|---|---|---|
| `domain` des articles sans URL rempli par « news.ycombinator.com », `url` laissée vide | ce sont des messages écrits directement sur Hacker News (86 % sont des Ask, Tell ou Launch HN) : leur contenu est hébergé sur le site lui-même. L'absence d'URL est une information, pas une erreur | 3 466 |

### Paramètres de suivi dans les URL

In [4]:
urls_nettoyees = articles["url"].map(retirer_parametres_de_suivi)
urls_modifiees = articles["url"].notna() & (urls_nettoyees != articles["url"])

print(f"URL modifiées : {urls_modifiees.sum()}")
print(f"Signatures AWS restantes : {urls_nettoyees.str.contains('X-Amz', na=False).sum()}")
pd.DataFrame({"avant": articles.loc[urls_modifiees, "url"], "après": urls_nettoyees[urls_modifiees]}).head()

URL modifiées : 204
Signatures AWS restantes : 0


,avant,après
483,https://cablehaunt.com/?,https://cablehaunt.com/
749,https://proton-native.js.org/#/v2_changes?,https://proton-native.js.org/#/v2_changes
757,https://pdf.sciencedirectassets.com/271600/1-s...,https://pdf.sciencedirectassets.com/271600/1-s...
792,https://blog.jonlu.ca/posts/ryan-air?ref=hnj20,https://blog.jonlu.ca/posts/ryan-air
838,https://docs.google.com/spreadsheets/d/1ehKtQe...,https://docs.google.com/spreadsheets/d/1ehKtQe...


In [5]:
articles["url"] = urls_nettoyees

| Quoi | Pourquoi | Lignes |
|---|---|---|
| paramètres de suivi retirés des URL (`utm_*`, `ref`, `source`, `fbclid`…) et signature AWS d'Elsevier (`X-Amz-*`) | ils ne disent rien du contenu et la signature AWS contient des identifiants (expirés) d'un tiers. Les autres paramètres sont gardés : sur 2 657 URL avec paramètres, la plupart en ont besoin, comme le `v=` des 849 vidéos YouTube | 204 |

### Articles republiés

In [6]:
articles = articles.sort_values("created_at").reset_index(drop=True)
articles["republication"] = articles["title"].duplicated()

print(f"Republications : {articles['republication'].sum()}")

Republications : 949


| Quoi | Pourquoi | Lignes |
|---|---|---|
| articles republiés gardés, signalés par une colonne `republication` | chaque publication a eu son propre score et ses propres commentaires : la supprimer fausserait l'activité du site. La colonne permet de compter les sujets distincts quand c'est utile | 949 |

### Scores et commentaires extrêmes

In [7]:
q1 = articles["score"].quantile(0.25)
q3 = articles["score"].quantile(0.75)
borne_haute = q3 + 1.5 * (q3 - q1)

print(f"Borne haute du score : {borne_haute:.0f}, articles au-dessus : {(articles['score'] > borne_haute).sum()}")
articles.nlargest(5, "score")[["title", "score", "num_comments", "created_at"]]

Borne haute du score : 554, articles au-dessus : 7640


,title,score,num_comments,created_at
65081,OpenAI's board has fired Sam Altman,5710,2530,2023-11-17 20:28:50+00:00
71042,Backdoor in upstream xz/liblzma leading to SSH...,4549,1849,2024-03-29 16:16:50+00:00
75362,CrowdStrike Update: Windows Bluescreen and Boo...,4489,3859,2024-07-19 05:26:13+00:00
60022,Bram Moolenaar has died,4310,428,2023-08-05 12:23:27+00:00
38204,Mechanical Watch,4298,413,2022-05-04 15:06:41+00:00


| Quoi | Pourquoi | Lignes |
|---|---|---|
| scores au-dessus de la borne haute (554 points) conservés | ce sont les grands événements de la période (renvoi de Sam Altman par OpenAI, porte dérobée dans xz, panne CrowdStrike) : des valeurs rares mais réelles. Les comparaisons utiliseront la médiane, moins sensible à ces valeurs | 7 640 |

### Articles sur l'IA

In [8]:
articles["parle_ia"] = parle_d_ia(articles["title"], articles["created_at"])

print(f"Articles sur l'IA : {articles['parle_ia'].sum()} ({articles['parle_ia'].mean():.1%})")
articles.loc[articles["parle_ia"], "title"].sample(8, random_state=0).tolist()

Articles sur l'IA : 5681 (5.7%)


['Thesis: Interesting work is less amenable to the use of AI',
 'What happened in this GPT-3 conversation?',
 'Agentic AI Foundation',
 'Clearview AI is struggling to address complaints as its legal issues mount',
 'OpenAI Leaks 120B Open Model on Hugging Face',
 'How AI is unlocking ancient texts',
 'Planting Undetectable Backdoors in Machine Learning Models',
 'Anthropic blocks third-party use of Claude Code subscriptions']

In [9]:
titres_shannon = articles["title"].str.contains("Claude Shannon")
titres_gemini_protocole = articles["title"].str.contains(r"\bGemini\b") & (articles["created_at"] < "2023-12-01")

print(f"Titres sur Claude Shannon comptés comme IA : {(titres_shannon & articles['parle_ia']).sum()} sur {titres_shannon.sum()}")
print(f"Titres sur le protocole Gemini comptés comme IA : {(titres_gemini_protocole & articles['parle_ia']).sum()} sur {titres_gemini_protocole.sum()}")

Titres sur Claude Shannon comptés comme IA : 2 sur 9
Titres sur le protocole Gemini comptés comme IA : 0 sur 13


| Quoi | Pourquoi | Lignes |
|---|---|---|
| colonne `parle_ia` ajoutée : le titre contient un mot-clé de l'IA (AI, LLM, ChatGPT, GPT-4, Claude, Llama, machine learning…) | c'est la variable centrale des questions 1 à 3. « Claude Shannon » est exclu, et « Gemini » n'est compté qu'à partir de décembre 2023 (lancement de Google Gemini) pour écarter le protocole internet du même nom. 2 des 9 titres sur Claude Shannon restent comptés : ils parlent bien de machine learning | 5 681 articles marqués |

## 2. Epoch AI

### Modèles sans date

In [10]:
nb_avant = len(modeles)
modeles = modeles.dropna(subset=["publication_date"])

print(f"Modèles retirés : {nb_avant - len(modeles)}, il en reste {len(modeles)}")

Modèles retirés : 12, il en reste 3614


| Quoi | Pourquoi | Lignes |
|---|---|---|
| modèles sans date de publication supprimés | toutes les questions sur les modèles portent sur leur évolution dans le temps : un modèle sans date ne peut pas y entrer | 12 |

### Colonnes redondantes

In [11]:
modeles = modeles.drop(columns=["training_compute_log10_flop", "publication_year", "last_modified"])

| Quoi | Pourquoi | Lignes |
|---|---|---|
| `training_compute_log10_flop`, `publication_year` et `last_modified` supprimées | les deux premières se recalculent à 100 % depuis `training_compute_flop` et `publication_date`. `last_modified` est la date de mise à jour de la fiche par Epoch AI, pas une information sur le modèle | 3 614 |

### Accès aux modèles

In [12]:
modeles["model_accessibility"] = modeles["model_accessibility"].fillna("Inconnu")
modeles.loc[modeles["model_accessibility"] == "Limited access", "open_model_weights"] = "No"

pd.crosstab(modeles["model_accessibility"], modeles["open_model_weights"].fillna("manquant"))

open_model_weights,No,Yes,manquant
model_accessibility,,,
API access,437,0,0
Hosted access (no API),133,0,0
Inconnu,0,0,832
Limited access,3,0,0
Open weights (non-commercial),0,229,0
Open weights (restricted use),0,301,0
Open weights (unrestricted),0,846,0
Unreleased,833,0,0


| Quoi | Pourquoi | Lignes |
|---|---|---|
| `model_accessibility` manquant remplacé par « Inconnu » | un quart des modèles n'a pas cette information : les supprimer retirerait trop de modèles de la question 4, et l'absence est elle-même une information | 832 |

| Quoi | Pourquoi | Lignes |
|---|---|---|
| `open_model_weights` mis à « No » pour les modèles en accès limité | « Limited access » veut dire que les poids ne sont pas publics, comme pour les autres catégories déjà à « No » | 3 |

### Plusieurs valeurs dans une cellule

In [13]:
separateurs = {"organization": ",", "country": ",", "organization_type": ";", "domain": ";"}

for colonne, separateur in separateurs.items():
    nettoyee = modeles[colonne].map(lambda valeur: normaliser_liste(valeur, separateur))
    lignes_modifiees = (modeles[colonne].notna() & (nettoyee != modeles[colonne])).sum()
    print(f"{colonne} : {lignes_modifiees} lignes modifiées, {modeles[colonne].nunique()} -> {nettoyee.nunique()} valeurs distinctes")
    modeles[colonne] = nettoyee

organization : 740 lignes modifiées, 1318 -> 1277 valeurs distinctes
country : 1011 lignes modifiées, 380 -> 176 valeurs distinctes
organization_type : 352 lignes modifiées, 30 -> 13 valeurs distinctes
domain : 459 lignes modifiées, 190 -> 117 valeurs distinctes


| Quoi | Pourquoi | Lignes |
|---|---|---|
| valeurs de `organization`, `country`, `organization_type` et `domain` dédoublonnées et triées dans chaque cellule | « United States of America,United States of America » devient « United States of America », « Industry;Academia » et « Academia;Industry » deviennent la même valeur. Le nombre de pays distincts passe de 380 à 176 | 740 / 1 011 / 352 / 459 |

### Organisations écrites de plusieurs façons

In [14]:
organisations_harmonisees = modeles["organization"].map(harmoniser_organisations)
lignes_modifiees = modeles["organization"].notna() & (organisations_harmonisees != modeles["organization"])

print(f"Lignes modifiées : {lignes_modifiees.sum()}")
pd.DataFrame({"avant": modeles.loc[lignes_modifiees, "organization"], "après": organisations_harmonisees[lignes_modifiees]}).value_counts().head(10)

Lignes modifiées : 585

avant                            après                      
Google DeepMind                  Google                         111
Meta AI                          Meta                            77
DeepMind                         Google                          45
Facebook AI Research             Meta                            30
Google Brain                     Google                          24
Microsoft Research               Microsoft                       21
Google Research                  Google                          18
Google DeepMind,Google Research  Google                           9
DeepMind,University of Oxford    Google,University of Oxford      8
Facebook AI                      Meta                             8
Name: count, dtype: int64

In [15]:
modeles["organization"] = organisations_harmonisees
modeles["organization"].str.split(",").explode().value_counts().head(10)

organization
Google                                         439
Meta                                           193
Microsoft                                      156
OpenAI                                         144
Alibaba                                        133
Nvidia                                         114
Stanford University                            112
Tsinghua University                            108
Massachusetts Institute of Technology (MIT)     88
Carnegie Mellon University (CMU)                81
Name: count, dtype: int64

| Quoi | Pourquoi | Lignes |
|---|---|---|
| les entités d'un même groupe regroupées sous un seul nom : Google (Google DeepMind, DeepMind, Google Brain, Google Research…), Meta (Meta AI, Facebook AI Research…), Microsoft (Microsoft Research…), Alibaba (Alibaba DAMO Academy) | la question 4 compare les labos : sans regroupement, Google serait éclaté en 7 noms. MetaMind, rachetée par Salesforce, n'est volontairement pas rattachée à Meta | 585 |

### Mesures manquantes et valeurs extrêmes

In [16]:
mesures = ["parameters", "training_compute_flop", "training_compute_cost_usd"]
(modeles[mesures].isna().mean() * 100).round(1)

parameters                   35.1
training_compute_flop        61.0
training_compute_cost_usd    93.8
dtype: float64

In [17]:
modeles.nlargest(5, "training_compute_flop")[["model_name", "organization", "publication_date", "training_compute_flop"]]

,model_name,organization,publication_date,training_compute_flop
3594,GPT-6 Astra,OpenAI,2026-09-03,1.000100e+27
3244,Grok 4,xAI,2025-07-09,5.000000e+26
2976,GPT-4.5,OpenAI,2025-02-27,3.800000e+26
2944,Grok 3,xAI,2025-02-17,3.500000e+26
3287,GPT-5,OpenAI,2025-08-07,6.600000e+25


| Quoi | Pourquoi | Lignes |
|---|---|---|
| puissance de calcul, paramètres et coût manquants non imputés | ces chiffres ne sont pas publiés, surtout pour les modèles fermés : les inventer fausserait justement les tendances étudiées. Les questions 4 et 5 porteront sur les modèles renseignés, en le précisant | 2 204 sans puissance de calcul |

| Quoi | Pourquoi | Lignes |
|---|---|---|
| valeurs extrêmes de puissance de calcul et de paramètres conservées | elles s'étalent naturellement sur des dizaines d'ordres de grandeur, de Theseus (1950) aux modèles de 2026. Elles seront étudiées en échelle logarithmique | 0 |

## 3. Enregistrement

In [18]:
articles.to_csv("../data/processed/articles.csv", index=False)
modeles.to_csv("../data/processed/modeles.csv", index=False)

print(f"Articles : {articles.shape}, modèles : {modeles.shape}")

Articles : (100286, 10), modèles : (3614, 25)
